# ДЗ 2. Dataset, DataLoader и корректная работа с данными

Задание к теме «Dataset/DataLoader, train/val/test split». На лекции вы видели `FashionMNIST`,
`ImageFolder`, `random_split`, `transforms.Compose` и базовый `DataLoader`. Здесь мы пойдём
дальше: `Dataset`/`DataLoader` — это общий механизм PyTorch для *любых* данных, не только
картинок, и большая часть реальных багов в ML-пайплайнах прячется именно на этом этапе — не в
архитектуре модели.

Все задачи используют синтетические или встроенные в `sklearn` данные (не требуют скачивания
и работают одинаково что в Colab, что офлайн), считаются секунды — минуты на CPU.


In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import numpy as np
torch.manual_seed(0)
np.random.seed(0)


## Уровень 1. Попроще

### Задача 1.1. Свой `Dataset` для табличных данных (с нормализацией и аугментацией)

Оберните любой табличный датасет (например, `sklearn.datasets.load_wine` — он не требует
скачивания) в собственный класс `NormalizingTabularDataset(Dataset)`:

1. В конструкторе примите сырые признаки `X` (numpy-массив) и метки `y`, посчитайте по всей
   выборке `X` среднее и стандартное отклонение каждого признака (z-score нормализация) и
   сохраните их как атрибуты объекта.
2. В `__getitem__` возвращайте нормализованный объект `(x - mean) / std` (в виде тензора) и
   соответствующую метку.
3. Добавьте параметры конструктора `train: bool = False` и `noise_std: float = 0.0`. Если
   `train=True` и `noise_std > 0`, добавляйте к нормализованному объекту гауссовский шум с этим
   стандартным отклонением (простая аугментация для табличных данных) — новый шум при каждом
   обращении к `__getitem__`. Если `train=False`, объект должен быть строго детерминирован
   (шум не добавляется независимо от значения `noise_std`).

Убедитесь, что `DataLoader` поверх вашего класса отдаёт батчи ожидаемой формы, и продемонстрируйте,
что при `train=True` повторные обращения к одному и тому же индексу дают разные тензоры, а при
`train=False` — совпадающие.


In [2]:
from sklearn.datasets import load_wine


class NormalizingTabularDataset(Dataset):
    def __init__(self, X, y, train, noise_std):
        super().__init__()
        self.X = X.astype(np.float32)
        self.y = y
        self.train = train
        self.noise_std = noise_std

        self.mean = torch.tensor(X.mean(axis=0), dtype=torch.float32)
        self.std = torch.tensor(X.std(axis=0), dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x_norm = (torch.tensor(self.X[idx]) - self.mean) / self.std

        if self.train and self.noise_std > 0:
            x_norm = x_norm + torch.randn_like(x_norm) * self.noise_std

        return x_norm, int(self.y[idx])


wine = load_wine()
X_raw, y_raw = wine.data, wine.target

train_ds = NormalizingTabularDataset(X_raw, y_raw, train=True, noise_std=0.1)
test_ds = NormalizingTabularDataset(X_raw, y_raw, train=False, noise_std=0.1)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=8, shuffle=False)

x_batch, y_batch = next(iter(train_loader))
print("Проверка формы батчей:")
print("x:", x_batch.shape)
print("y:", y_batch.shape)


idx = 0

tr1 = train_ds[idx][0]
tr2 = train_ds[idx][0]

print("\nПроверка на train:")
print("1:", tr1.numpy())
print("2:", tr2.numpy())
print("Равны?", torch.allclose(tr1, tr2))

te1 = test_ds[idx][0]
te2 = test_ds[idx][0]
print("\nПроверка на test:")
print("1:", te1.numpy())
print("2:", te2.numpy())
print("Равны?", torch.allclose(te1, te2))

Проверка формы батчей:
x: torch.Size([8, 13])
y: torch.Size([8])

Проверка на train:
1: [ 1.2513555  -0.56538296  0.2819315  -1.2219253   1.8887576   0.7034441
  0.97889274 -0.67153394  1.2085294   0.22667053  0.25335464  1.8483605
  1.0362878 ]
2: [ 1.4365978  -0.6301516   0.37456122 -0.97311807  1.9906895   1.0079882
  1.1066442  -0.7092297   1.3558868   0.29127255  0.37112504  1.8310987
  0.96573305]
Равны? False

Проверка на test:
1: [ 1.518612   -0.5622497   0.23205288 -1.169593    1.9139055   0.80899733
  1.0348189  -0.65956306  1.2248839   0.25171685  0.3621771   1.8479198
  1.013009  ]
2: [ 1.518612   -0.5622497   0.23205288 -1.169593    1.9139055   0.80899733
  1.0348189  -0.65956306  1.2248839   0.25171685  0.3621771   1.8479198
  1.013009  ]
Равны? True


### Задача 1.2. Стратифицированный train/test split без искажения баланса классов

Представьте, что среди объектов один класс сильно **редкий** (например, при бинарной
классификации редких событий: мошеннические транзакции, редкие заболевания и т.п.). Если
разбивать выборку на train/test **совершенно случайно** (без учёта меток), при небольшом размере
теста доля редкого класса в train и test может случайно сильно разойтись с исходной — вплоть до
того, что в test не попадёт вообще ни одного объекта редкого класса, и оценить качество модели на
нём станет невозможно.

1. Сгенерируйте синтетические метки: 1000 объектов, 3 класса с сильно разными частотами
   (например, 800/150/50).
2. Реализуйте `stratified_split(labels, test_frac)`, который возвращает индексы train и test так,
   чтобы **доля каждого класса** в train и в test была одинаковой (с точностью до округления) и
   равнялась доле этого класса во всей выборке — то есть сплит делается независимо **внутри
   каждого класса**, а не по всей выборке целиком.
3. Продемонстрируйте на этом же массиве `labels`, что наивный `sklearn.model_selection.
   train_test_split` без параметра `stratify` (или ручной случайный сплит по всем объектам сразу)
   может дать заметно смещённые пропорции классов в test, особенно для редкого класса: запустите
   наивный сплит несколько раз с разными `random_state` и посчитайте, насколько сильно доля
   редкого класса в test отклоняется от его доли в полной выборке — в среднем и в худшем случае.


In [3]:
from sklearn.model_selection import train_test_split


def stratified_split(labels, test_frac):

    classes = np.unique(labels)
    train_idx = []
    test_idx = []

    for cl in classes:
        cls_indices = np.where(labels == cl)[0]

        rng = np.random.default_rng()
        shuffled = rng.permutation(cls_indices)

        n_test = int(len(shuffled) * test_frac)

        test_idx.append(shuffled[:n_test])
        train_idx.append(shuffled[n_test:])

    return np.concatenate(train_idx), np.concatenate(test_idx)


np.random.seed(42)
n_samples = 1000
labels = np.array([0]*800 + [1]*150 + [2]*50)
np.random.shuffle(labels)

true_ratios = {}
for c in range(3):
    true_ratios[c] = np.mean(labels == c)

print("Истинные доли классов:", true_ratios)

train_idx, test_idx = stratified_split(labels, test_frac=0.2)
test_labels_strat = labels[test_idx]

strat_ratios = {}
for c in range(3):
    strat_ratios[c] = np.mean(test_labels_strat == c)

print("\nСтратифицированный сплит")
for c in range(3):
    diff = abs(strat_ratios[c] - true_ratios[c])
    print(f"Класс {c}: истинная={true_ratios[c]:.3f}, в test={strat_ratios[c]:.3f}, "
          f"отклонение={diff:.3f}")

indices = np.arange(n_samples)

_, tts_idx_sample, _, tts_labels_sample = train_test_split(
    indices,
    labels,
    test_size=0.2,
    random_state=72
)

tts_labels = labels[tts_idx_sample]
tts_ratios = {}
for c in range(3):
    tts_ratios[c] = np.mean(tts_labels == c)

print("\ntrain_test_split:")
for c in range(3):
    diff = abs(tts_ratios[c] - true_ratios[c])
    print(f"Класс {c}: истинная={true_ratios[c]:.3f}, в tts={tts_ratios[c]:.3f}, "
          f"отклонение={diff:.3f}")

deviations_rare = []
for seed in range(100):
    _, test_naive = train_test_split(
        labels,
        test_size=0.2,
        random_state=seed
    )
    rare_ratio_in_test = np.mean(test_naive == 2)
    deviations_rare.append(abs(rare_ratio_in_test - true_ratios[2]))

avg_deviation = np.mean(deviations_rare)
max_dev = np.max(deviations_rare)

print("\nСреднее отклонение доли редкого класса:", np.round(avg_deviation, 3))
print("Худшее отклонение доли редкого класса:", np.round(max_dev, 3))

Истинные доли классов: {0: np.float64(0.8), 1: np.float64(0.15), 2: np.float64(0.05)}

Стратифицированный сплит
Класс 0: истинная=0.800, в test=0.800, отклонение=0.000
Класс 1: истинная=0.150, в test=0.150, отклонение=0.000
Класс 2: истинная=0.050, в test=0.050, отклонение=0.000

train_test_split:
Класс 0: истинная=0.800, в tts=0.805, отклонение=0.005
Класс 1: истинная=0.150, в tts=0.155, отклонение=0.005
Класс 2: истинная=0.050, в tts=0.040, отклонение=0.010

Среднее отклонение доли редкого класса: 0.011
Худшее отклонение доли редкого класса: 0.03


### Задача 1.3. `WeightedRandomSampler` для многоклассового дисбаланса

Постройте синтетический **трёхклассовый** датасет с сильным дисбалансом (700 объектов класса 0,
200 — класса 1, 100 — класса 2). Реализуйте функцию `make_balancing_weights(labels)`, которая для
тензора или массива меток **произвольного** числа классов возвращает тензор весов для каждого
объекта, обратно пропорциональных частоте его класса (без хардкода числа классов или конкретных
частот — функция должна одинаково работать и на 2, и на 5 классах). Сравните распределение классов
в батчах обычного `DataLoader(shuffle=True)` и `DataLoader` с
`sampler=WeightedRandomSampler(weights=make_balancing_weights(labels), ...)`. Посчитайте среднюю
долю **каждого** класса по нескольким батчам в обоих случаях и убедитесь, что с весами все три
класса представлены в батчах примерно поровну.


In [4]:
from torch.utils.data import WeightedRandomSampler


def make_balancing_weights(labels):
    classes, counts = torch.unique(labels, return_counts=True)

    class_weights = 1.0 / counts.to(torch.float64)

    weight_lookup = torch.zeros(len(classes), dtype=torch.float64)
    weight_lookup[classes] = class_weights
    sample_weights = weight_lookup[labels]
    return sample_weights


np.random.seed(42)
n_samples = 1000
labels_list = np.array([0]*700 + [1]*200 + [2]*100)
np.random.shuffle(labels_list)
labels = torch.tensor(labels_list, dtype=torch.long)


class SimpleDataset(Dataset):
    def __init__(self, labels):
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return torch.tensor([idx], dtype=torch.float32), self.labels[idx]


dataset = SimpleDataset(labels)
weights = make_balancing_weights(labels)

loader = DataLoader(dataset, batch_size=64, shuffle=True)

sampler = WeightedRandomSampler(
    weights=weights,
    num_samples=len(weights),
    replacement=True
)
loader_weighted = DataLoader(dataset, batch_size=64, sampler=sampler)


def get_class_distribution(loader, n_batches):
    all_labels = []
    for i, (_, batch_labels) in enumerate(loader):
        if i >= n_batches:
            break
        all_labels.append(batch_labels)

    all_labels = torch.cat(all_labels)
    classes = torch.unique(all_labels).sort()[0].tolist()
    ratios = {}
    for c in classes:
        ratios[c] = (all_labels == c).float().mean().item()
    return ratios


print("Истинное распределение:")
counts = {0: 700, 1: 200, 2: 100}

true_ratios = {}
for c in counts:
    true_ratios[c] = counts[c] / n_samples

for c, r in true_ratios.items():
    print(f"Класс {c}: {r:.3f}")

print("\nDataLoader (shuffle=True):")
dl_ratios = get_class_distribution(loader, 50)
for c, r in dl_ratios.items():
    print(f"Класс {c}: {r:.3f}")

print("\nWeightedRandomSampler:")
weighted_ratios = get_class_distribution(loader_weighted, 50)
for c, r in weighted_ratios.items():
    print(f"Класс {c}: {r:.3f}")

Истинное распределение:
Класс 0: 0.700
Класс 1: 0.200
Класс 2: 0.100

DataLoader (shuffle=True):
Класс 0: 0.700
Класс 1: 0.200
Класс 2: 0.100

WeightedRandomSampler:
Класс 0: 0.330
Класс 1: 0.344
Класс 2: 0.326


## Уровень 2. Посложнее

### Задача 2. Кастомный `collate_fn` для пар параллельных последовательностей переменной длины

Во многих реальных задачах объект — это не одна последовательность, а **несколько параллельных**
последовательностей одинаковой (для данного объекта) длины: например, последовательность токенов
текста и последовательность длительностей пауз между ними, или последовательность ID событий и
последовательность времени с предыдущего события. При паддинге такого объекта важно паддить обе
последовательности **одной и той же** длиной и использовать **одну и ту же** маску.

1. Реализуйте `PairSeqDataset`, возвращающий тройки: (целочисленная последовательность случайной
   длины $L$ — «токены», вещественная последовательность той же длины $L$ — «временные
   интервалы», бинарная метка).
2. Реализуйте `pad_pair_collate(batch)` — функцию для параметра `collate_fn` в `DataLoader`,
   которая паддит обе последовательности нулями до длины максимальной именно в этом батче и
   возвращает: паддинговый тензор токенов, паддинговый тензор временных интервалов, общую маску
   реальных (не паддинговых) позиций, исходные длины и метки.
3. Убедитесь, что: (а) число реальных позиций, посчитанное по маске, совпадает с суммой исходных
   длин последовательностей; (б) маска, применённая к паддинговому тензору токенов и к
   паддинговому тензору интервалов, действительно обнуляет одни и те же позиции в обоих (то есть
   паддинг синхронизирован между двумя последовательностями).


In [12]:
# TODO: PairSeqDataset + pad_pair_collate

rnd_num = 1
np.random.seed(rnd_num)
torch.manual_seed(rnd_num)

class PairSeqDataset(Dataset):
    def __init__(self, n_samples=100, min_len=3, max_len=10, n_tokens=100):
        self.n_samples = n_samples
        self.min_len = min_len
        self.max_len = max_len
        self.n_tokens = n_tokens
        
        # Генерируем данные заранее
        self.data = []
        for _ in range(n_samples):
            # Случайная длина последовательности
            length = np.random.randint(min_len, max_len + 1)
            
            # Последовательность токенов (целые числа от 0 до n_tokens-1)
            tokens = np.random.randint(0, n_tokens, size=length).tolist()
            
            # Последовательность временных интервалов (вещественные числа)
            intervals = np.random.uniform(0.1, 2.0, size=length).tolist()
            
            # Бинарная метка
            label = np.random.randint(0, 2)
            
            self.data.append((tokens, intervals, label))
    
    def __len__(self):
        return self.n_samples
    
    def __getitem__(self, idx):
        tokens, intervals, label = self.data[idx]
        # Возвращаем как тензоры
        return (
            torch.tensor(tokens, dtype=torch.long),
            torch.tensor(intervals, dtype=torch.float32),
            torch.tensor(label, dtype=torch.long)
        )


def pad_pair_collate(batch):
    # Извлекаем компоненты батча
    tokens_list = [item[0] for item in batch]
    intervals_list = [item[1] for item in batch]
    labels_list = [item[2] for item in batch]
    
    # Находим максимальную длину в батче
    lengths = [len(t) for t in tokens_list]
    max_len = max(lengths)
    
    batch_size = len(batch)
    
    # Создаём паддинговые тензоры, заполненные нулями
    padded_tokens = torch.zeros(batch_size, max_len, dtype=torch.long)
    padded_intervals = torch.zeros(batch_size, max_len, dtype=torch.float32)
    mask = torch.zeros(batch_size, max_len, dtype=torch.float32)
    
    # Заполняем тензоры реальными данными и создаём маску
    for i, (tokens, intervals) in enumerate(zip(tokens_list, intervals_list)):
        seq_len = len(tokens)
        
        # Копируем реальные данные
        padded_tokens[i, :seq_len] = tokens
        padded_intervals[i, :seq_len] = intervals
        
        # Отмечаем реальные позиции в маске
        mask[i, :seq_len] = 1.0
    
    # Собираем метки в тензор
    labels = torch.stack(labels_list)
    
    return padded_tokens, padded_intervals, mask, lengths, labels


# Создаём датасет
dataset = PairSeqDataset(n_samples=20, min_len=3, max_len=8)

# Создаём DataLoader с нашим collate_fn
dataloader = DataLoader(dataset, batch_size=4, shuffle=False, collate_fn=pad_pair_collate)

# Берём первый батч
for batch_idx, (padded_tokens, padded_intervals, mask, lengths, labels) in enumerate(dataloader):
    if batch_idx >= 1:  # Проверяем только первый батч
        break
    for i in range(len(lengths)):
        print(f"\nОбъект {i}:")
        print(f"  Исходная длина: {lengths[i]}")
        print(f"  Padded tokens: {padded_tokens[i].tolist()}")
        print(f"  Padded intervals: {[f'{x:.2f}' for x in padded_intervals[i].tolist()]}")
        print(f"  Mask: {mask[i].tolist()}")
        print(f"  Label: {labels[i].item()}")



Объект 0:
  Исходная длина: 8
  Padded tokens: [12, 72, 9, 75, 5, 79, 64, 16]
  Padded intervals: ['0.85', '0.84', '1.37', '1.88', '1.71', '0.70', '1.10', '0.94']
  Mask: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
  Label: 0

Объект 1:
  Исходная длина: 8
  Padded tokens: [14, 50, 68, 87, 87, 94, 96, 86]
  Padded intervals: ['1.62', '1.94', '0.70', '1.42', '1.77', '1.80', '0.26', '0.17']
  Mask: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
  Label: 1

Объект 2:
  Исходная длина: 5
  Padded tokens: [72, 30, 71, 3, 70, 0, 0, 0]
  Padded intervals: ['1.92', '1.11', '1.41', '0.70', '1.40', '0.00', '0.00', '0.00']
  Mask: [1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0]
  Label: 0

Объект 3:
  Исходная длина: 3
  Padded tokens: [41, 82, 15, 0, 0, 0, 0, 0]
  Padded intervals: ['1.36', '0.67', '0.95', '0.00', '0.00', '0.00', '0.00', '0.00']
  Mask: [1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0]
  Label: 1


---
**Что сдавать:** ноутбук с реализованными классами/функциями, пройденными проверками и
письменными ответами на вопросы для решённых задач. Всё задание в сумме считается на CPU не
дольше нескольких минут.
